# Regional Housing Market Dashboard
**Data Source:** U.S. Census Bureau — ACS 5-Year Estimates (2022)  
**States:** Minnesota, Wisconsin, Illinois, Iowa, Michigan  
**Author:** Scott A. May

This notebook pulls county-level housing data from the Census API and builds an interactive four-chart dashboard comparing home values, income, and occupancy rates across five Upper Midwest states.

In [9]:
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")
YEAR = 2022

STATES = {
    "Minnesota":  "27",
    "Wisconsin":  "55",
    "Illinois":   "17",
    "Iowa":       "19",
    "Michigan":   "26",
}

VARIABLES = "NAME,B25077_001E,B19013_001E,B25002_003E,B25003_002E,B25003_003E"

print("Configuration loaded ✓")

Configuration loaded ✓


## Step 1: Pull Data from Census API
Pulling county-level housing data for all five states.

In [10]:
print("Pulling Census data for all states...")
all_data = []

for state_name, state_fips in STATES.items():
    url = (
        f"https://api.census.gov/data/{YEAR}/acs/acs5"
        f"?get={VARIABLES}"
        f"&for=county:*"
        f"&in=state:{state_fips}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df["state_name"] = state_name
    all_data.append(df)
    print(f"  ✓ {state_name} — {len(df)} counties")

df = pd.concat(all_data, ignore_index=True)
print(f"\nTotal counties loaded: {len(df)}")

Pulling Census data for all states...
  ✓ Minnesota — 87 counties
  ✓ Wisconsin — 72 counties
  ✓ Illinois — 102 counties
  ✓ Iowa — 99 counties
  ✓ Michigan — 83 counties

Total counties loaded: 443


## Step 2: Clean and Shape the Data
Renaming columns, converting to numeric, and calculating occupancy rates.

In [11]:
df = df.rename(columns={
    "B25077_001E": "median_home_value",
    "B19013_001E": "median_income",
    "B25002_003E": "vacant_units",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
})

for col in ["median_home_value","median_income","vacant_units",
            "owner_occupied","renter_occupied"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["median_home_value","median_income"])
df["county_name"] = df["NAME"].str.split(",").str[0]
df["renter_pct"] = (
    df["renter_occupied"] /
    (df["owner_occupied"] + df["renter_occupied"]) * 100
).round(1)
df["owner_pct"] = (100 - df["renter_pct"]).round(1)

print(f"Data cleaned ✓ — {len(df)} counties")
df.head()

Data cleaned ✓ — 443 counties


,NAME,median_home_value,median_income,vacant_units,owner_occupied,renter_occupied,state,county,state_name,county_name,renter_pct,owner_pct
0,"Aitkin County, Minnesota",222100,56406,7400,5751,1046,27,001,Minnesota,Aitkin County,15.4,84.6
1,"Anoka County, Minnesota",302300,95782,4143,107811,26579,27,003,Minnesota,Anoka County,19.8,80.2
2,"Becker County, Minnesota",249600,68683,5539,11049,3085,27,005,Minnesota,Becker County,21.8,78.2
3,"Beltrami County, Minnesota",203900,62173,3533,12114,5756,27,007,Minnesota,Beltrami County,32.2,67.8
4,"Benton County, Minnesota",229300,70346,1006,10842,5470,27,009,Minnesota,Benton County,33.5,66.5


## Step 3: Build State-Level Summary
Aggregating county data to state-level medians for comparison charts.

In [12]:
state_summary = df.groupby("state_name").agg(
    avg_home_value  = ("median_home_value", "median"),
    avg_income      = ("median_income",     "median"),
    avg_renter_pct  = ("renter_pct",        "mean"),
    avg_owner_pct   = ("owner_pct",         "mean"),
    county_count    = ("county_name",       "count"),
).reset_index().round(1)

state_summary = state_summary.sort_values("avg_home_value", ascending=True)

print("State summary ✓")
state_summary

State summary ✓


,state_name,avg_home_value,avg_income,avg_renter_pct,avg_owner_pct,county_count
0,Illinois,122000.0,63573.0,24.6,75.4,102
1,Iowa,141500.0,66056.0,24.4,75.6,99
2,Michigan,155900.0,59467.0,20.5,79.5,83
4,Wisconsin,194100.0,67761.5,24.5,75.5,72
3,Minnesota,203900.0,69396.0,22.1,77.9,87


## Step 4: Build the Dashboard
Four-chart interactive dashboard comparing home values, income, and occupancy rates across states.

In [13]:
STATE_COLORS = {
    "Minnesota": "#2597F5",
    "Wisconsin":  "#4CAF50",
    "Illinois":   "#FF5722",
    "Iowa":       "#9C27B0",
    "Michigan":   "#FF9800",
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Median Home Value by State",
        "Owner vs. Renter Rate by State",
        "Home Value vs. Median Income by County",
        "Top 20 Counties by Median Home Value",
    ),
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

# Chart 1: Median Home Value by State
fig.add_trace(
    go.Bar(
        x=state_summary["avg_home_value"],
        y=state_summary["state_name"],
        orientation="h",
        marker_color=[STATE_COLORS[s] for s in state_summary["state_name"]],
        text=[f"${v:,.0f}" for v in state_summary["avg_home_value"]],
        textposition="outside",
        showlegend=False,
    ),
    row=1, col=1,
)

# Chart 2: Owner vs Renter Rate
fig.add_trace(
    go.Bar(
        name="Owner Occupied %",
        x=state_summary["state_name"],
        y=state_summary["avg_owner_pct"],
        marker_color="#42A5F5",
        text=[f"{v:.1f}%" for v in state_summary["avg_owner_pct"]],
        textposition="outside",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        name="Renter Occupied %",
        x=state_summary["state_name"],
        y=state_summary["avg_renter_pct"],
        marker_color="#EF5350",
        text=[f"{v:.1f}%" for v in state_summary["avg_renter_pct"]],
        textposition="outside",
    ),
    row=1, col=2,
)

# Chart 3: Scatter — Home Value vs Income
for state_name, group in df.groupby("state_name"):
    fig.add_trace(
        go.Scatter(
            x=group["median_income"],
            y=group["median_home_value"],
            mode="markers",
            name=state_name,
            marker=dict(color=STATE_COLORS[state_name], size=7, opacity=0.7),
            text=group["county_name"],
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Income: $%{x:,.0f}<br>"
                "Home Value: $%{y:,.0f}<extra></extra>"
            ),
        ),
        row=2, col=1,
    )

# Chart 4: Top 20 Counties
top20 = df.nlargest(20, "median_home_value").sort_values("median_home_value", ascending=True)
fig.add_trace(
    go.Bar(
        x=top20["median_home_value"],
        y=top20["county_name"] + ", " + top20["state_name"],
        orientation="h",
        marker_color=[STATE_COLORS[s] for s in top20["state_name"]],
        text=[f"${v:,.0f}" for v in top20["median_home_value"]],
        textposition="outside",
        showlegend=False,
    ),
    row=2, col=2,
)

fig.update_layout(
    title=dict(
        text="Midwest Regional Housing Market Dashboard — 2022 ACS Data",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    height=900,
    barmode="group",
    plot_bgcolor="#F8F9FA",
    paper_bgcolor="#FFFFFF",
    font=dict(family="Arial", size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.update_xaxes(title_text="Median Home Value ($)", row=1, col=1)
fig.update_xaxes(title_text="State", row=1, col=2)
fig.update_xaxes(title_text="Median Household Income ($)", row=2, col=1)
fig.update_xaxes(title_text="Median Home Value ($)", row=2, col=2)
fig.update_yaxes(title_text="Median Home Value ($)", row=2, col=1)

fig.show()

## Notes
- Data sourced from the U.S. Census Bureau ACS 5-Year Estimates (2022)
- State-level values represent median aggregations of county-level data
- Renter/owner percentages calculated from occupied housing units only
- Counties with missing home value or income data excluded from analysis